## **Aim**
To simulate the process of "file carving" by scanning a binary blob for specific file signatures (magic numbers) to recover "deleted" data.

## **Algorithm**
**Step 1:** Create a "disk image" binary file that contains random data, interspersed with actual file content (e.g., a small JPEG) and then marked as "deleted" (by simply not having a file system entry).

**Step 2:** Define the magic number for the file type to be recovered (e.g., JPEG: `\xFF\xD8\xFF`).

**Step 3:** Define an "end-of-file" marker or a fixed recovery size.

**Step 4:** Read the disk image byte-by-byte to find the start signature.

**Step 5:** Once found, extract data from that point until the end marker or fixed limit.

**Step 6:** Save the extracted bytes as a recovered file.

In [1]:
import os

def carve_files(disk_image, target_sig, recovered_prefix):
    with open(disk_image, "rb") as f:
        data = f.read()
    
    # Find all occurrences of the signature
    start_index = 0
    count = 0
    
    while True:
        start_index = data.find(target_sig, start_index)
        if start_index == -1:
            break
        
        # For simulation, we extract a fixed block of 100 bytes as the "file"
        # In real carving, you'd look for a footer (e.g., \xFF\xD9 for JPEG)
        end_index = start_index + 100
        carved_data = data[start_index:end_index]
        
        filename = f"{recovered_prefix}_{count}.bin"
        with open(filename, "wb") as recovered_file:
            recovered_file.write(carved_data)
        
        print(f"Recovered file found at offset {start_index}: {filename}")
        count += 1
        start_index += 1 # Move forward to find next
        
    return count

def main():
    disk_image = "simulated_disk.bin"
    jpeg_sig = b"\xff\xd8\xff"
    
    # Create a simulated disk with 'deleted' files
    with open(disk_image, "wb") as f:
        f.write(os.urandom(500)) # Random noise
        f.write(b"\xff\xd8\xff" + b"Fake JPEG Data 1" + b"X"*50) # "Deleted" file 1
        f.write(os.urandom(300)) # Random noise
        f.write(b"\xff\xd8\xff" + b"Fake JPEG Data 2" + b"X"*50) # "Deleted" file 2
        f.write(os.urandom(200))
        
    print(f"Scanning {disk_image} for JPEG signatures...")
    recovered_count = carve_files(disk_image, jpeg_sig, "recovered_img")
    print(f"\nTotal files recovered: {recovered_count}")

if __name__ == "__main__":
    main()

Scanning simulated_disk.bin for JPEG signatures...
Recovered file found at offset 500: recovered_img_0.bin
Recovered file found at offset 869: recovered_img_1.bin

Total files recovered: 2


## **Result**
This the program successfully simulates the recovery of deleted files by scanning for byte patterns.